# S12 pratimas

```
S12 pratimas -- ataskaitų teikimas: PRISMA srautas ir DI atsekamumas.

Užduotis
--------
Duota galutinė tiriamosios (scoping) apžvalgos būsena
(``examples/chapter/data/pipeline_state_fixture.json``) ir DI naudojimo
atsekamumo ataskaita (``ai_traceability_fixture.json``).

1. Įgyvendinkite ``prisma_flow(state)`` -- sudarykite PRISMA srauto skaičius
   ir patikrinkite penkias tvermės lygčių grupes bei tiriamojo režimo
   tinkamumo patikrą (kokybės vertinimas praleistas). Jei būsena nėra
   galutinė (netuščia pilno teksto peržiūros eilė) -- iškelkite klaidą.
2. Įgyvendinkite ``classify_stage_oversight(...)`` -- kiekvienam etapui
   grąžinkite tris nepriklausomus matmenis: ``execution`` (ar vyko),
   ``automation_origin`` (kas automatizavo) ir ``human_reviewed`` (ar
   peržiūrėjo žmogus).
3. Paaiškinkite žodyno skirtumą tarp „vienas etapas automatizuotas" ir
   „visiška (viso pipeline'o) autonomija".

Paleidimas: ``python -m examples.chapter.s12_reporting.exercise``
```

1. Paleiskite paruošimo langelį. 2. Užpildykite `TODO` funkcijas pratimo langelyje ir paleiskite jį. 3. Palyginkite su pavyzdiniu sprendimu.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rimraul/slr-automation.git"
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
for req in ['examples/chapter/requirements-colab.txt']:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("pip install nepavyko (zr. klaida aukščiau): " + req)
print("Paruosta:", os.getcwd())

In [ ]:

from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, ConfigDict


class PrismaInconsistencyError(ValueError):
    pass


class StageOversight(BaseModel):
    model_config = ConfigDict(frozen=True)

    execution: Literal["run", "not_run", "unknown_or_not_instrumented"]
    automation_origin: Literal["ai", "rule_based_automation"] | None
    human_reviewed: bool


class PrismaFlow(BaseModel):
    total_identified: int
    duplicates_removed: int
    after_dedup: int
    screened_title_abstract: int
    excluded_title_abstract: int
    included_after_ta: int
    fulltext_sought: int
    assessed_full_text: int
    acquisition_failures: int
    excluded_full_text: int
    excluded_full_text_reasons: dict[str, int]
    included: int

    def to_ascii(self) -> str:
        raise NotImplementedError


def prisma_flow(state: dict) -> PrismaFlow:
    """TODO: sudarykite srautą ir patikrinkite tvermės grupes."""
    raise NotImplementedError


def classify_stage_oversight(traceability: dict, prisma_counts: dict, oversight_mode: dict) -> dict:
    """TODO: trys nepriklausomi priežiūros matmenys kiekvienam etapui."""
    raise NotImplementedError


def main() -> None:
    from examples.chapter import toolkit

    state = toolkit.load_fixture("pipeline_state_fixture.json")
    print("S12 reporting exercise")
    print(f"review type: {state['review_type']}; FT review queue size: {len(state['full_text_review_queue'])}")
    try:
        flow = prisma_flow(state)
    except NotImplementedError:
        print("prisma_flow() is not implemented yet -- fill it in and re-run.")
        return
    print(flow.to_ascii())


main()

## Pavyzdinis sprendimas

Pažiūrėkite tik pabandę patys. Žemiau esantys testai tikrina **pavyzdinį sprendimą**, ne jūsų kodą.

In [ ]:
!cat examples/chapter/s12_reporting/solution.py

In [ ]:
!python -m pytest examples/chapter/s12_reporting -q

In [ ]:
# NEPRIVALOMA: tikri kreipiniai i DKM (sumoka uz naudojimą). Raktą laikykite
# Colab "Secrets" (rakto piktograma kairėje), niekada ne užrašykite kode.
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")